# Pyspark Data Engineering — v3 Validation

This notebook validates data consistency, Q62–Q64 edge cases, and the presence/syntax of all 80 solutions. Runtime and full correctness require separate execution of the solutions notebook.


In [ ]:
from pathlib import Path
import csv, json, re, ast
from collections import Counter
from IPython.display import display
import pandas as pd
DATA = Path('data')
assert DATA.is_dir(), f'Missing data directory: {DATA.resolve()}'


In [ ]:
def read_csv(name):
    with (DATA / f'{name}.csv').open(newline='', encoding='utf-8-sig') as f:
        return list(csv.DictReader(f))
def read_json(name):
    return json.loads((DATA / f'{name}.json').read_text(encoding='utf-8'))
def read_jsonl(name):
    return [json.loads(line) for line in (DATA / f'{name}.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
customers = read_csv('customers')
transactions = read_csv('transactions')
checks = []
def check(label, condition, details=''):
    checks.append({'test':label,'status':'PASS' if condition else 'FAIL','details':details})
check('Customer count unchanged',len(customers)==120,str(len(customers)))
check('Transaction count unchanged',len(transactions)==400,str(len(transactions)))
check('Q62 blank email coverage',sum(not (r['email'] or '').strip() for r in customers)>=3)
valid_emails=[(r['email'] or '').strip().lower() for r in customers if (r['email'] or '').strip()]
check('Q64 duplicate normalized email coverage',any(v>1 for v in Counter(valid_emails).values()))
check('Q63 invalid amount coverage',sum(not r['amount'] or float(r['amount'])<=0 for r in transactions)>=3)
for name in ['customers','transactions']:
    a=read_csv(name); b=read_json(name); c=read_jsonl(name)
    check(f'{name} format row counts',len(a)==len(b)==len(c),f'csv={len(a)} json={len(b)} jsonl={len(c)}')
    idcol='customer_id' if name=='customers' else 'transaction_id'
    def normalize_id(value):
        if value is None or str(value).strip() == "":
            return None
        return int(value)

    csv_ids = {normalize_id(r[idcol]) for r in a}
    json_ids = {normalize_id(r[idcol]) for r in b}
    jsonl_ids = {normalize_id(r[idcol]) for r in c}

    check(
        f"{name} IDs match across formats",
        csv_ids == json_ids == jsonl_ids
    )
# Q62 — Validate all three missing-email scenarios independently

emails = [r.get("email") for r in read_json("customers")]

# 1. NULL email
check(
    "Q62 NULL email",
    any(email is None for email in emails)
)

# 2. Empty string email
check(
    "Q62 empty email",
    any(email == "" for email in emails)
)

# 3. Whitespace-only email
check(
    "Q62 whitespace-only email",
    any(
        isinstance(email, str)
        and email != ""
        and email.strip() == ""
        for email in emails
    )
)
check('Q63 JSON explicit null',any(r['amount'] is None for r in read_json('transactions')))


In [ ]:
nb_path=next(Path('.').glob('03_*questions_and_solutions.ipynb'))
nb=json.loads(nb_path.read_text(encoding='utf-8'))
# Solution notebook structure is alternating markdown and executable code cells.
solutions=[]
for i,c in enumerate(nb['cells']):
    if c['cell_type']!='markdown':continue
    title=''.join(c['source'])
    m=re.search(r'\bQ(\d{1,2})\b',title)
    if not m:continue
    j=i+1
    while j<len(nb['cells']) and nb['cells'][j]['cell_type']!='code':j+=1
    if j<len(nb['cells']):
        source=''.join(nb['cells'][j]['source'])
        if source.strip():solutions.append((int(m.group(1)),source))
ids={i for i,_ in solutions}
check('All 80 solutions present',set(range(1,81)).issubset(ids),f'Found {len(ids)} unique Q IDs')
syntax_errors=[]
for i,src in solutions:
    try:ast.parse(src)
    except SyntaxError as e:syntax_errors.append((i,str(e)))
check('Solution Python syntax',not syntax_errors,str(syntax_errors[:5]))
result=pd.DataFrame(checks)
display(result)
print(result.status.value_counts().to_dict())
result.to_csv('validation_v3_report.csv',index=False)
assert (result.status=='PASS').all(), 'One or more v3 checks failed.'
